In [2]:
"""
Save Pipeline Outputs — All Modules (1→2→3→4)
═══════════════════════════════════════════════════════════════════════════════
Project  : Automatic Road Network Extraction with Connectivity and
           Topology Refinement from High-Resolution Satellite Imagery
Dataset  : SpaceNet 5 (SN5) — AOI 8, Mumbai, India
Institute: VR Siddhartha Engineering College, Vijayawada
Team     : Murala Leela Kartheek (228W1A0529)
           Mallam Manoj         (228W1A0525)
           Ch Devarshisai       (228W1A0510)
Guide    : Dr. G. Kranthi Kumar

PRIMARY OUTPUT (enabled):
  pipeline_outputs/
    ├── all_patches/      ← 5-column combined: Input|GT|M2|M3|M4
    └── summary_grid.png  ← compact 15-patch grid for project report

INDIVIDUAL MODULE OUTPUTS (commented out):
  # module1_patches/     ← normalized image patches
  # module2_masks/       ← binary mask + yellow overlay
  # module3_centerlines/ ← red centerline overlay
  # module4_graphs/      ← blue graph overlay
═══════════════════════════════════════════════════════════════════════════════
"""

import numpy as np
import cv2
import math
import json
import matplotlib
matplotlib.use("Agg")   # non-interactive — faster, no display needed
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")


# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════

CONFIG = {
    # Input paths
    "val_npz":         "/home/jupyter-228w1a0529/Major/Dataset-1/processed/val.npz",
    "centerlines_npz": "/home/jupyter-228w1a0529/Major/Dataset-1/processed/centerlines/centerlines.npz",
    "vectors_dir":     "/home/jupyter-228w1a0529/Major/Dataset-1/processed/vectors",

    # Output root
    "output_dir":      "/home/jupyter-228w1a0529/Major/pipeline_outputs",

    # Number of patches to save
    # None = save ALL patches (~3971, takes ~45 min)
    # int  = save N random patches (recommended: 100–200)
    "n_patches":       30,

    # Summary grid rows (max 15 recommended for readability)
    "summary_rows":    15,

    # Image quality
    "dpi": 100,

    # ImageNet denormalization
    "mean": np.array([0.485, 0.456, 0.406]),
    "std":  np.array([0.229, 0.224, 0.225]),
}


# ═══════════════════════════════════════════════════════════════════════════
# LOAD ALL DATA
# ═══════════════════════════════════════════════════════════════════════════

def load_all_data(config=CONFIG):
    """Load val patches, centerlines, and Module 4 GeoJSON graphs."""
    print("[INFO] Loading data...")

    val_data    = np.load(config["val_npz"])
    images      = val_data["images"]     # (N, 3, H, W) float32
    gt_masks    = val_data["masks"]      # (N, H, W)    float32

    cl_data     = np.load(config["centerlines_npz"])
    centerlines = cl_data["centerlines"] # (N, H, W) uint8
    eval_masks  = cl_data["eval_masks"]  # (N, H, W) uint8 — Module 3 output
    raw_preds   = cl_data["raw_preds"]   # (N, H, W) uint8 — Module 2 output

    print(f"[INFO] Loaded {len(images)} patches")

    # Reconstruct Module 4 graphs from saved GeoJSON
    vectors_dir = Path(config["vectors_dir"])
    graphs = []
    if vectors_dir.exists():
        print("[INFO] Loading Module 4 graphs from GeoJSON...")
        for i in tqdm(range(len(images)), desc="Loading graphs"):
            gj_path = vectors_dir / f"patch_{i:04d}.geojson"
            if gj_path.exists():
                with open(gj_path) as f:
                    gj = json.load(f)
                graphs.append(_geojson_to_graph(gj))
            else:
                graphs.append(nx.Graph())
    else:
        print("[WARN] No Module 4 vectors found — graph column will be blank")
        graphs = [nx.Graph() for _ in range(len(images))]

    return images, gt_masks, raw_preds, eval_masks, centerlines, graphs


def _geojson_to_graph(geojson: dict) -> nx.Graph:
    """Reconstruct NetworkX graph from saved GeoJSON for visualization."""
    G = nx.Graph()
    for feat in geojson.get("features", []):
        coords = feat["geometry"]["coordinates"]
        if len(coords) < 2:
            continue
        pts = [(int(c[1]), int(c[0])) for c in coords]
        for i in range(len(pts) - 1):
            u, v = pts[i], pts[i+1]
            G.add_node(u, y=u[0], x=u[1])
            G.add_node(v, y=v[0], x=v[1])
            length = math.sqrt((u[0]-v[0])**2 + (u[1]-v[1])**2)
            G.add_edge(u, v, length=length, geometry=[u, v])
    return G


# ═══════════════════════════════════════════════════════════════════════════
# DRAW HELPERS
# ═══════════════════════════════════════════════════════════════════════════

def denorm(image_chw: np.ndarray, config=CONFIG) -> np.ndarray:
    """(3,H,W) normalised float32 → (H,W,3) float32 for display."""
    img = np.transpose(image_chw, (1, 2, 0))
    return (img * config["std"] + config["mean"]).clip(0, 1)


def draw_graph_overlay(img_hwc: np.ndarray, G: nx.Graph) -> np.ndarray:
    """Blue edges + green normal nodes + red junction nodes on image."""
    ov   = img_hwc.copy()
    H, W = ov.shape[:2]
    for u, v, data in G.edges(data=True):
        geom = data.get("geometry", [u, v])
        for i in range(len(geom) - 1):
            r1,c1 = int(geom[i][0]),   int(geom[i][1])
            r2,c2 = int(geom[i+1][0]), int(geom[i+1][1])
            if 0<=r1<H and 0<=c1<W and 0<=r2<H and 0<=c2<W:
                cv2.line(ov, (c1,r1), (c2,r2), (0.0,0.5,1.0), 1)
    for node in G.nodes():
        r, c = int(node[0]), int(node[1])
        if 0 <= r < H and 0 <= c < W:
            color = (1.0,0.1,0.1) if G.degree(node)>=3 else (0.1,1.0,0.1)
            cv2.circle(ov, (c, r), 2, color, -1)
    return ov


# ═══════════════════════════════════════════════════════════════════════════
# ── INDIVIDUAL MODULE SAVES (COMMENTED OUT) ───────────────────────────────
# Uncomment any section below if you want per-module individual images
# ═══════════════════════════════════════════════════════════════════════════

def save_module1_patch(idx, img, gt, out_dir, config=CONFIG):
    """Module 1 — data preparation: image + GT mask + overlay."""
    img_d = denorm(img, config)
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img_d);             axes[0].set_title("Satellite image"); axes[0].axis("off")
    axes[1].imshow(gt, cmap="gray");   axes[1].set_title("GT road mask");    axes[1].axis("off")
    ov = img_d.copy(); ov[gt.astype(bool)] = [1.0,0.9,0.0]
    axes[2].imshow(ov);                axes[2].set_title("GT overlay");      axes[2].axis("off")
    plt.suptitle(f"Module 1 — Data Preparation | Patch {idx:04d}", fontsize=10, fontweight="bold")
    plt.tight_layout()
    plt.savefig(out_dir / f"patch_{idx:04d}.png", dpi=config["dpi"], bbox_inches="tight")
    plt.close()


def save_module2_mask(idx, img, gt, raw_pred, out_dir, config=CONFIG):
    """Module 2 — segmentation: input | GT | prediction | overlay."""
    img_d = denorm(img, config)
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(img_d);                  axes[0].set_title("Input image");     axes[0].axis("off")
    axes[1].imshow(gt, cmap="gray");        axes[1].set_title("Ground truth");    axes[1].axis("off")
    axes[2].imshow(raw_pred, cmap="gray");  axes[2].set_title("Predicted mask");  axes[2].axis("off")
    ov = img_d.copy(); ov[raw_pred.astype(bool)] = [1.0,0.9,0.0]
    axes[3].imshow(ov);                     axes[3].set_title("Yellow overlay");  axes[3].axis("off")
    inter = float((raw_pred.astype(bool) & gt.astype(bool)).sum())
    union = float((raw_pred.astype(bool) | gt.astype(bool)).sum())
    plt.suptitle(f"Module 2 — Segmentation | Patch {idx:04d} | IoU={inter/(union+1e-6):.3f}", fontsize=10, fontweight="bold")
    plt.tight_layout()
    plt.savefig(out_dir / f"patch_{idx:04d}.png", dpi=config["dpi"], bbox_inches="tight")
    plt.close()


def save_module3_centerline(idx, img, gt, raw_pred, eval_mask, centerline, out_dir, config=CONFIG):
    """Module 3 — connectivity: input | GT | M2 mask | closed mask | red centerline."""
    img_d = denorm(img, config)
    ov = img_d.copy()
    k  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3))
    cl = cv2.dilate(centerline.astype(np.uint8),k).astype(bool)
    ov[cl] = [1.0,0.15,0.15]
    fig, axes = plt.subplots(1, 5, figsize=(22, 4))
    axes[0].imshow(img_d);               axes[0].set_title("Input image");       axes[0].axis("off")
    axes[1].imshow(gt, cmap="gray");     axes[1].set_title("Ground truth");      axes[1].axis("off")
    axes[2].imshow(raw_pred,cmap="gray");axes[2].set_title("Module 2 raw");      axes[2].axis("off")
    axes[3].imshow(eval_mask,cmap="gray");axes[3].set_title("After closing");    axes[3].axis("off")
    axes[4].imshow(ov);                  axes[4].set_title("Centerline (red)");  axes[4].axis("off")
    plt.suptitle(f"Module 3 — Connectivity | Patch {idx:04d}", fontsize=10, fontweight="bold")
    plt.tight_layout()
    plt.savefig(out_dir / f"patch_{idx:04d}.png", dpi=config["dpi"], bbox_inches="tight")
    plt.close()


def save_module4_graph(idx, img, gt, centerline, G, out_dir, config=CONFIG):
    """Module 4 — vectorization: input | GT | centerline | graph overlay."""
    img_d = denorm(img, config)
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    axes[0].imshow(img_d);               axes[0].set_title("Input image");     axes[0].axis("off")
    axes[1].imshow(gt, cmap="gray");     axes[1].set_title("Ground truth");    axes[1].axis("off")
    axes[2].imshow(centerline,cmap="gray");axes[2].set_title("M3 centerline"); axes[2].axis("off")
    gr_ov = draw_graph_overlay(img_d, G)
    bp = mpatches.Patch(color=(0,.5,1),   label="Road edge")
    gp = mpatches.Patch(color=(0.1,1,0.1),label="Node")
    rp = mpatches.Patch(color=(1,0.1,0.1),label="Junction")
    axes[3].imshow(gr_ov)
    axes[3].legend(handles=[bp,gp,rp], loc="lower right", fontsize=7)
    axes[3].set_title(f"G=(V={G.number_of_nodes()}, E={G.number_of_edges()})"); axes[3].axis("off")
    plt.suptitle(f"Module 4 — Vectorization | Patch {idx:04d}", fontsize=10, fontweight="bold")
    plt.tight_layout()
    plt.savefig(out_dir / f"patch_{idx:04d}.png", dpi=config["dpi"], bbox_inches="tight")
    plt.close()


# ═══════════════════════════════════════════════════════════════════════════
# COMBINED 5-COLUMN OUTPUT (PRIMARY — ENABLED)
# Columns: Input | Ground Truth | Module 2 | Module 3 | Module 4
# ═══════════════════════════════════════════════════════════════════════════

def save_combined_patch(idx, img, gt, raw_pred, eval_mask,
                         centerline, G, out_dir, config=CONFIG):
    """
    Save a single image showing all 4 module outputs side by side.

    Layout (5 columns):
      Col 1 — Input satellite image
      Col 2 — Ground truth road mask
      Col 3 — Module 2: DeepLabV3+ binary prediction
      Col 4 — Module 3: Morphologically refined eval mask
      Col 5 — Module 4: Vectorized road network G=(V,E) overlaid on image

    This is the PRIMARY output — gives the complete pipeline story
    in a single image, ideal for project reports and presentations.
    """
    img_d    = denorm(img, config)
    gr_ov    = draw_graph_overlay(img_d, G)

    fig, axes = plt.subplots(1, 5, figsize=(22, 4))

    # ── Col 1: Input ──────────────────────────────────────────────────────
    axes[0].imshow(img_d)
    axes[0].set_title("Input\n(Satellite PS-RGB)", fontsize=8,
                       fontweight="bold")
    axes[0].axis("off")

    # ── Col 2: Ground truth ───────────────────────────────────────────────
    axes[1].imshow(gt, cmap="gray")
    axes[1].set_title("Ground Truth\n(GT road mask)", fontsize=8,
                       fontweight="bold")
    axes[1].axis("off")

    # ── Col 3: Module 2 output ────────────────────────────────────────────
    axes[2].imshow(raw_pred, cmap="gray")
    axes[2].set_title("Module 2\n(DeepLabV3+ segmentation)", fontsize=8,
                       fontweight="bold")
    axes[2].axis("off")

    # ── Col 4: Module 3 output ────────────────────────────────────────────
    axes[3].imshow(eval_mask, cmap="gray")
    axes[3].set_title("Module 3\n(Connectivity + centerline)", fontsize=8,
                       fontweight="bold")
    axes[3].axis("off")

    # ── Col 5: Module 4 output ────────────────────────────────────────────
    bp = mpatches.Patch(color=(0, 0.5, 1),    label="Road edge")
    gp = mpatches.Patch(color=(0.1, 1, 0.1),  label="Node")
    rp = mpatches.Patch(color=(1, 0.1, 0.1),  label="Junction")
    axes[4].imshow(gr_ov)
    axes[4].legend(handles=[bp, gp, rp], loc="lower right",
                    fontsize=6, framealpha=0.7)
    n_n = G.number_of_nodes()
    n_e = G.number_of_edges()
    axes[4].set_title(
        f"Module 4\nRoad network G=(V={n_n}, E={n_e})",
        fontsize=8, fontweight="bold"
    )
    axes[4].axis("off")

    plt.suptitle(
        f"Full Pipeline — Patch {idx:04d}  |  "
        f"SpaceNet 5 AOI-8 Mumbai",
        fontsize=10, fontweight="bold", y=1.01
    )
    plt.tight_layout()
    plt.savefig(out_dir / f"patch_{idx:04d}.png",
                dpi=config["dpi"], bbox_inches="tight")
    plt.close()


# ═══════════════════════════════════════════════════════════════════════════
# SUMMARY GRID — compact multi-patch grid for project report
# ═══════════════════════════════════════════════════════════════════════════

def save_summary_grid(indices, images, gt_masks, raw_preds,
                       eval_masks, centerlines, graphs,
                       save_path, config=CONFIG):
    """
    Compact N×5 grid showing all pipeline stages for N patches.
    Each row = one patch, each column = one pipeline stage.

    This is the BEST single image to include in your project report.
    Use summary_rows=8–15 for a clean presentation image.
    """
    n_rows     = min(len(indices), config["summary_rows"])
    col_titles = [
        "Input\n(Satellite)",
        "Ground Truth\n(GT mask)",
        "Module 2\n(Segmentation)",
        "Module 3\n(Connectivity)",
        "Module 4\n(Road Network G=(V,E))"
    ]

    fig = plt.figure(figsize=(22, n_rows * 3.5))

    for row, idx in enumerate(indices[:n_rows]):
        img_d = denorm(images[idx], config)
        gr_ov = draw_graph_overlay(img_d, graphs[idx])

        col_data = [
            (img_d,                None),
            (gt_masks[idx],        "gray"),
            (raw_preds[idx],       "gray"),
            (eval_masks[idx],      "gray"),
            (gr_ov,                None),
        ]

        for col, (data, cmap) in enumerate(col_data):
            ax = fig.add_subplot(n_rows, 5, row * 5 + col + 1)
            ax.imshow(data, cmap=cmap)

            # Column headers on first row only
            if row == 0:
                ax.set_title(col_titles[col], fontsize=9,
                             fontweight="bold", pad=5)

            # Patch label on leftmost column
            if col == 0:
                ax.set_ylabel(f"P{idx:04d}", fontsize=8,
                              rotation=0, labelpad=32, va="center")
            ax.axis("off")

    plt.suptitle(
        "Full Pipeline Summary — Automatic Road Network Extraction\n"
        "SpaceNet 5 (AOI-8 Mumbai)  |  M1→M2→M3→M4",
        fontsize=13, fontweight="bold", y=1.005
    )
    plt.tight_layout(h_pad=0.4, w_pad=0.4)
    plt.savefig(save_path, dpi=config["dpi"], bbox_inches="tight")
    plt.close()
    print(f"[INFO] Saved summary grid ({n_rows} patches) → {save_path}")


# ═══════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════

def save_all_outputs(config=CONFIG):
    """
    Save pipeline outputs.

    PRIMARY (enabled):
      all_patches/      — one combined 5-column image per patch
      summary_grid.png  — compact multi-patch grid for report

    INDIVIDUAL (commented out — uncomment if needed):
      module1_patches/
      module2_masks/
      module3_centerlines/
      module4_graphs/
    """
    print("=" * 62)
    print(" Saving Pipeline Outputs — M1 → M2 → M3 → M4")
    print("=" * 62)

    # Load all data
    images, gt_masks, raw_preds, eval_masks, centerlines, graphs = \
        load_all_data(config)

    N = len(images)

    # Select patches to save
    if config["n_patches"] is None:
        indices = list(range(N))
    else:
        np.random.seed(42)
        indices = sorted(
            np.random.choice(N, min(config["n_patches"], N),
                              replace=False).tolist()
        )
    print(f"[INFO] Saving {len(indices)} patches out of {N} total")

    # Create output directories
    out_root = Path(config["output_dir"])
    out_root.mkdir(parents=True, exist_ok=True)

    # ── PRIMARY: combined all-patch directory ──────────────────────────────
    all_dir = out_root / "all_patches"
    all_dir.mkdir(exist_ok=True)

    # ── INDIVIDUAL MODULE DIRECTORIES (commented out) ──────────────────────
    m1_dir = out_root / "module1_patches";    m1_dir.mkdir(exist_ok=True)
    m2_dir = out_root / "module2_masks";      m2_dir.mkdir(exist_ok=True)
    m3_dir = out_root / "module3_centerlines";m3_dir.mkdir(exist_ok=True)
    m4_dir = out_root / "module4_graphs";     m4_dir.mkdir(exist_ok=True)

    print(f"\n[INFO] Output root  : {out_root}")
    print(f"[INFO] Combined dir : {all_dir}/")
    print()

    # ── Save combined 5-column image for each patch ────────────────────────
    for idx in tqdm(indices, desc="Saving combined images"):
        img = images[idx]
        gt  = gt_masks[idx].astype(np.uint8)
        rp  = raw_preds[idx].astype(np.uint8)
        em  = eval_masks[idx].astype(np.uint8)
        cl  = centerlines[idx].astype(np.uint8)
        G   = graphs[idx]

        # ── PRIMARY: combined 5-column image ──────────────────────────────
        save_combined_patch(idx, img, gt, rp, em, cl, G, all_dir, config)

        # ── INDIVIDUAL MODULE OUTPUTS (commented out) ─────────────────────
        save_module1_patch(idx, img, gt, m1_dir, config)
        save_module2_mask(idx, img, gt, rp, m2_dir, config)
        save_module3_centerline(idx, img, gt, rp, em, cl, m3_dir, config)
        save_module4_graph(idx, img, gt, cl, G, m4_dir, config)

    # ── Save summary grid for project report ──────────────────────────────
    print("\n[INFO] Saving summary grid for project report...")
    save_summary_grid(
        indices,
        images, gt_masks, raw_preds, eval_masks, centerlines, graphs,
        save_path=str(out_root / "summary_grid.png"),
        config=config
    )

    # Final report
    n = len(indices)
    print(f"\n{'='*62}")
    print(f" ✅  Done!")
    print(f"{'='*62}")
    print(f"  all_patches/     : {n} combined 5-column images")
    print(f"  summary_grid.png : 1 compact grid ({config['summary_rows']} patches)")
    print(f"{'='*62}")
    print(f"\n  📋 Use summary_grid.png in your project report!")
    print(f"  📂 All outputs at: {out_root}/")

    return out_root


# ═══════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ═══════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    out_dir = save_all_outputs(CONFIG)

 Saving Pipeline Outputs — M1 → M2 → M3 → M4
[INFO] Loading data...
[INFO] Loaded 3971 patches
[INFO] Loading Module 4 graphs from GeoJSON...


Loading graphs: 100%|██████████| 3971/3971 [00:00<00:00, 4766.05it/s]


[INFO] Saving 30 patches out of 3971 total

[INFO] Output root  : /home/jupyter-228w1a0529/Major/pipeline_outputs
[INFO] Combined dir : /home/jupyter-228w1a0529/Major/pipeline_outputs/all_patches/



Saving combined images: 100%|██████████| 30/30 [00:45<00:00,  1.53s/it]



[INFO] Saving summary grid for project report...
[INFO] Saved summary grid (15 patches) → /home/jupyter-228w1a0529/Major/pipeline_outputs/summary_grid.png

 ✅  Done!
  all_patches/     : 30 combined 5-column images
  summary_grid.png : 1 compact grid (15 patches)

  📋 Use summary_grid.png in your project report!
  📂 All outputs at: /home/jupyter-228w1a0529/Major/pipeline_outputs/
